# 五会话路径风险 mark 时间顺序纠正

旧四例卡的股数/现金/费用/终端 NAV 机械身份保留。先核其计划/结果和旧代码 SHA，再用同四个来源时钟按实际分钟顺序重放。修正只影响可用 mark 高水位与回撤，不读新价值目标，也不改变事件选择。

In [1]:
import os,json,hashlib,warnings
from pathlib import Path
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
warnings.filterwarnings("ignore",message="The 'generic' unit for NumPy timedelta")
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_checkpoint import capture_after_open
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
from src.backtest.ibkr_tiered import settle_tiered_event
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from copy import deepcopy
root=Path.cwd();card=root/"research/stage1_v5_lock_risk_mark_order_correction_20261030";plan=json.loads((card/"plan.json").read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"]
assert sha(old/"plan.json")==plan["old_source_plan_sha256"]
assert sha(root/plan["checkpoint_code"])==plan["checkpoint_code_sha256"]
assert sha(root/plan["minute_provider_uri"]/"qlib_store_manifest.json")==plan["minute_provider_manifest_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/plan["minute_provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{y}" for y in parent_plan["signal_years"]],*[f"models/recovery_{y}" for y in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==1056 and book["audit"]["status"]=="passed"
open_models={y:DatedModel(objects[f"models/open_{y}"],f"{y-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{y}") for y in parent_plan["signal_years"]}
recovery_models={}
for y in parent_plan["signal_years"]:
 if y in parent_plan["explicit_no_recovery_model_years"]:recovery_models[y]=DatedModel(None,f"{y-1}-12-31",f"explicit_no_model_{y}_insufficient_prior_teacher",explicit_no_model=True)
 else:recovery_models[y]=DatedModel(objects[f"models/recovery_{y}"],f"{y-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{y}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
print('dated parent and minute source verified',len(receipt))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[26469:MainThread](2026-10-30 14:00:01,346) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[26469:MainThread](2026-10-30 14:00:01,347) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[26469:MainThread](2026-10-30 14:00:01,347) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[26469:MainThread](2026-10-30 14:00:01,435) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[26469:MainThread](2026-10-30 14:00:01,436) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[26469:MainThread](2026-10-30 14:00:01,436) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_event_guard_fullminute_20261028')}


dated parent and minute source verified 13


In [2]:
from src.backtest.stage1_capfree_lock_episode import replay_capfree_lock_episode
by_date={};quotes_by_date={};read_count=0
for item in plan["cases"]:
 day=pd.Timestamp(item["trade_date"])
 if day in by_date:continue
 checkpoint=capture_after_open(frame,event_day=day,controller_by_date=controller,
     recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],plan=parent_plan)
 by_date[day]=checkpoint
 last=frame.iloc[checkpoint.index+4]
 rawbars=D.features(["SOXL","SOXX"],["$open","$close","$raw_open","$price_valid"],
     start_time=str(day.date())+" 09:31:00",end_time=str(pd.Timestamp(last.trade_date).date())+" 16:00:00",freq="1min")
 parts=[]
 for symbol in ("SOXL","SOXX"):
  x=rawbars.xs(symbol,level="instrument").copy();x.columns=[symbol+"_"+v.removeprefix("$") for v in x.columns];parts.append(x)
 quotes_by_date[day]=pd.concat(parts,axis=1).sort_index()
 read_count+=1
assert read_count==plan["budget"]["qlib_minute_reads"]
print('Qlib source reads',read_count)

Qlib source reads 3


In [3]:
results=[];episodes={}
for item in plan["cases"]:
 day=pd.Timestamp(item["trade_date"]);checkpoint=by_date[day];quotes=quotes_by_date[day]
 bar_end=pd.Timestamp(str(day.date())+' '+item["bar_end_at"]+':00')
 fill=bar_end+pd.Timedelta(minutes=2);decision=bar_end+pd.Timedelta(minutes=1)
 late=bar_end>day+pd.Timedelta(hours=10,minutes=32)
 pos=book["native_positions"].get(day+pd.Timedelta(hours=10,minutes=32))
 assert pos is not None
 recovery_orders=book["orders"].loc[(book["orders"].trade_date==day)&(book["orders"].slot=='recovery')].to_dict('records')
 recovery_state=None
 if late:
  recovery_state={"quantities":np.array([pos.get_stock_amount(s) if pos.check_stock(s) else 0. for s in ('SOXL','SOXX')],float),"cash":float(pos.get_cash()),"orders":recovery_orders}
 mark=quotes.loc[bar_end];mark_price=np.array([mark.SOXL_close,mark.SOXX_close],float)
 q=recovery_state["quantities"] if late else checkpoint.state.q
 cash=recovery_state["cash"] if late else checkpoint.state.cash
 nav=float(q@mark_price+cash);w=np.r_[q*mark_price,cash]/nav
 if item['target_rule']=='cash_to_soxx_0.10':target=w+np.array([0.,.10,-.10])
 elif item['target_rule']=='soxl_minus_0.04_soxx_plus_0.08_cash_minus_0.04':target=w+np.array([-.04,.08,-.04])
 else:raise ValueError('blocked: target rule absent')
 fillbar=quotes.loc[fill+pd.Timedelta(minutes=1)];adjusted=np.array([fillbar.SOXL_open,fillbar.SOXX_open],float);raw_fill=np.array([fillbar.SOXL_raw_open,fillbar.SOXX_raw_open],float)
 event={'trade_date':day,'bar_end_at':bar_end,'decision_at':decision,'fill_at':fill,'desired_weights':target,'adjusted_fill_prices':adjusted,'raw_fill_prices':raw_fill}
 out=replay_capfree_lock_episode(frame,checkpoint,quotes,event=event,recovery_state=recovery_state,horizon_sessions=5)
 hold=out['HOLD'];branch=out['EVENT'];terminal=frame.iloc[checkpoint.index+4][['soxl_exit_open','soxx_exit_open']].to_numpy(float)
 delta=float(branch['path'].iloc[-1].next_open_nav-hold['path'].iloc[-1].next_open_nav)
 direct=float((branch['ending_quantities']-hold['ending_quantities'])@terminal+branch['ending_cash']-hold['ending_cash'])
 assert abs(delta-direct)<1e-7 and branch['ending_cash']>=-1e-8
 assert branch['path'].event_fee_usd.iloc[1:].eq(0).all() and hold['path'].event_fee_usd.eq(0).all()
 expected_extra=sum(int(o['raw_shares']) for o in branch['orders'] if not o['cap_applied'])
 month=fill.strftime('%Y-%m');actual_extra=branch['monthly_fee_shares'].get(month,0)-hold['monthly_fee_shares'].get(month,0)
 assert actual_extra==expected_extra
 parent_next=float(book['independent_replay'].iloc[checkpoint.index].native_expected_nav)
 hold_parent_error=abs(float(hold['path'].iloc[0].next_open_nav)-parent_next)
 if not book['independent_replay'].iloc[checkpoint.index].recovery_action or late:assert hold_parent_error<1e-7
 if late:
  assert out['post_1032_parent_state_spliced']
  assert np.max(np.abs(out['prefill_quantities']-recovery_state['quantities']))==0
  assert abs(out['prefill_cash']-recovery_state['cash'])==0
 else:assert not out['post_1032_parent_state_spliced']
 episodes[item['name']]=(checkpoint,quotes,event,out)
 results.append({'case':item['name'],'day':str(day.date()),'clock':str(bar_end)+'->'+str(fill),'session_close':str(out['session_close']),'recovery_orders_before_decision':len(recovery_orders) if late else 0,'post_1032_parent_state_spliced':out['post_1032_parent_state_spliced'],'event_order_count':len(branch['orders']),'event_order_symbols':[o['symbol'] for o in branch['orders']],'event_order_sell':[bool(o['sell']) for o in branch['orders']],'event_fee_usd':float(sum(o['total'] for o in branch['orders'])),'monthly_fee_share_delta':actual_extra,'event_status':out['event_status'],'hold_parent_first_next_open_error_usd':hold_parent_error,'terminal_nav_closure_error_usd':abs(delta-direct),'hold_available_mark_max_drawdown':float(hold['path'].available_mark_max_drawdown.max()),'event_available_mark_max_drawdown':float(branch['path'].available_mark_max_drawdown.max()),'hold_terminal_available_mark_max_drawdown':float(hold['terminal_available_mark_max_drawdown']),'event_terminal_available_mark_max_drawdown':float(branch['terminal_available_mark_max_drawdown'])})

old_card=root/plan['old_card']
assert sha(old_card/'plan.json')==plan['old_plan_sha256'] and sha(old_card/'audit.json')==plan['old_audit_sha256']
old_audit=json.loads((old_card/'audit.json').read_text())
assert old_audit['module_sha256']==plan['old_module_sha256']
old_cases={row['case']:row for row in old_audit['cases']}
for current in results:
 prior=old_cases[current['case']]
 for field in ('event_order_count','event_order_symbols','event_order_sell','event_status','monthly_fee_share_delta'):
  assert current[field]==prior[field]
 for field in ('event_fee_usd','hold_parent_first_next_open_error_usd','terminal_nav_closure_error_usd'):
  assert abs(current[field]-prior[field])<1e-7
# Independent chronological reconstruction for the ordinary event: sort source
# marks by actual local ET time, with next-open inserted after its 09:47 bar.
checkpoint,quotes,event,out=episodes['ordinary_early']
independent={}
for arm in ('HOLD','EVENT'):
 q=out[arm]['ending_quantities'];cash=out[arm]['ending_cash']
 sequence=[];day=pd.Timestamp(event['trade_date']);first=quotes.loc[quotes.index.normalize()==day]
 for stamp in first.loc[(first.index>=day+pd.Timedelta(hours=9,minutes=48)) & (first.index<=event['fill_at'])].index:
  if stamp<=event['bar_end_at']:
   q0=checkpoint.state.q;c0=checkpoint.state.cash
  else:
   q0=checkpoint.state.q;c0=checkpoint.state.cash
  sequence.append((stamp,0,float(q0@first.loc[stamp,['SOXL_close','SOXX_close']].to_numpy(float)+c0)))
 for offset in range(5):
  row=frame.iloc[checkpoint.index+offset];session=pd.Timestamp(row.trade_date)
  part=quotes.loc[quotes.index.normalize()==session]
  if offset==0:part=part.loc[part.index>=event['fill_at']+pd.Timedelta(minutes=1)]
  for stamp,bar in part.iterrows():
   sequence.append((stamp,0,float(q@bar[['SOXL_close','SOXX_close']].to_numpy(float)+cash)))
  next_day=pd.Timestamp(row.exit_date)
  next_nav=float(q@row[['soxl_exit_open','soxx_exit_open']].to_numpy(float)+cash)
  sequence.append((next_day+pd.Timedelta(hours=9,minutes=47),1,next_nav))
 sequence.sort(key=lambda row:(row[0],row[1]))
 nav=np.array([x[2] for x in sequence],float)
 peak=np.maximum.accumulate(np.r_[checkpoint.state.peak,nav])[1:]
 risk=float(np.max(1-nav/peak))
 independent[arm]=risk
 assert abs(risk-out[arm]['terminal_available_mark_max_drawdown'])<1e-12
assert len(results)==len(plan['cases'])==4
audit={'experiment':plan['experiment'],'plan_sha256':sha(card/'plan.json'),'old_module_sha256':plan['old_module_sha256'],'corrected_module_sha256':sha(root/plan['module']),'old_audit_sha256':plan['old_audit_sha256'],'cases':results,'independent_chronological_risk_ordinary':independent,'orders_cash_fee_terminal_nav_unchanged':True,'new_model_fits':0,'new_candidate_accounts':0,'historical_research_passed':False,'decision':'risk_mark_order_corrected_terminal_nav_parity_preserved'}
(card/'audit.json').write_text(json.dumps(audit,ensure_ascii=False,indent=2)+'\n')
print(json.dumps(audit,ensure_ascii=False,indent=2))

{
  "experiment": "stage1_v5_lock_risk_mark_order_correction_20261030",
  "plan_sha256": "04520fd465e96310d544fc7e3d8308dfd79f8632b0695894c87c494271c18f0c",
  "old_module_sha256": "1b3d8988db90a29ddc5700759cc4aebabfc32ddd3d126fc84188fa6a1d1fd3c1",
  "corrected_module_sha256": "537e2aea5ecf1c9c493f937f6cc67fbd788440a7169d321a88b3b9eb08be1725",
  "old_audit_sha256": "2a287c0211a38a4cc6002265097e7033f2aa0b98ecf9373474097c8053c706fe",
  "cases": [
    {
      "case": "ordinary_early",
      "day": "2019-12-19",
      "clock": "2019-12-19 09:48:00->2019-12-19 09:50:00",
      "session_close": "2019-12-19 16:00:00",
      "recovery_orders_before_decision": 0,
      "post_1032_parent_state_spliced": false,
      "event_order_count": 1,
      "event_order_symbols": [
        "SOXX"
      ],
      "event_order_sell": [
        false
      ],
      "event_fee_usd": 0.55524925,
      "monthly_fee_share_delta": 64.0,
      "event_status": "funded",
      "hold_parent_first_next_open_error_usd": 0.

## 中文结论

SOXX硬门：零模型、零候选账户、零0/2/5bp检验。此前单事件引擎把每个次日09:47开盘mark提前放在当日09:31—09:47分钟bar之前，高水位/最大回撤时序不合格。现按真实先后处理早盘bar→09:47开盘→其后bar，五会话终端开盘在最后收盘之后；逐日风险列止于本会话收盘，终端风险另列。四个旧Qlib机械案例的订单、费用、现金和终端NAV误差未变；四例的最大回撤数值也恰好未变，不能据此为旧算法签发通用资格。独立按时间戳排序的普通案例HOLD/EVENT风险与新实现逐项相同。合成回归：先80后120的次日，正确最大回撤20%，旧顺序将其算成33.33%；聚焦测试通过。

旧卡保留原模块SHA和原结果，当前模块SHA以本纠正卡为准。还没有全分钟路径标签、模型信息门或SOXX账户；新批量风险计算必须使用修正后的时间序列，并独立核对分钟/隔夜价格完整性与五会话成熟。